# Exploring Ship Traffic with AIS Data

**Prerequisites**

- Polars introduction

**Outcomes**

- Learn about AIS data and its role as "alternative data" in economics
- Practice using Polars to load, filter, and aggregate large datasets
- Identify locations where cargo ships stop for extended periods
- Use DBSCAN clustering to discover port locations from raw ship data

**Data**

- [Marine Cadastre AIS Data](https://hub.marinecadastre.gov/pages/vesseltraffic) published by the US Government

In [ ]:
import datetime as dt

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from shapely.geometry import LineString, Point

%matplotlib inline
plt.style.use("ggplot")

# Load coastline data (used for map backgrounds)
world = gpd.read_file(
    "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
)

## What is AIS Data?

AIS stands for **Automatic Identification System**. It is an onboard navigation safety device that transmits and monitors the location and characteristics of vessels in US and international waters in real time.

The US Coast Guard collects these transmissions through a national network of receivers. Several government agencies (the Bureau of Ocean Energy Management, NOAA, and the Coast Guard) have worked together to make a cleaned version of this data available to the public through [Marine Cadastre](https://hub.marinecadastre.gov/pages/vesseltraffic).

### Why should economists care about ship data?

AIS data is an example of **alternative data** -- non-traditional datasets that can provide timely, even real-time, snapshots of economic activity.

Traditional economic indicators like GDP, trade balances, and unemployment are published with significant lags (weeks to months). Ship traffic data, by contrast, is available almost immediately. Researchers at the IMF demonstrated this in a [working paper](https://www.imf.org/en/publications/wp/issues/2020/05/14/world-seaborne-trade-in-real-time-a-proof-of-concept-for-building-ais-based-nowcasts-from-49393) where they used AIS data to build "nowcasts" of world seaborne trade.

The core idea: if you can observe where ships are stopping and for how long, you can learn something about the volume and flow of global trade *before* official statistics are released.

### The data dictionary

Each row in our dataset is a single transmission from a ship at a particular moment in time. The key columns are:

| Field | Description | Example |
|-------|-------------|--------|
| `mmsi` | Maritime Mobile Service Identity -- a unique ship identifier | 477220100 |
| `base_date_time` | UTC timestamp of the transmission | 2024-01-01T05:02:00 |
| `latitude` | Latitude in decimal degrees | 42.35137 |
| `longitude` | Longitude in decimal degrees | -71.04182 |
| `sog` | Speed Over Ground in knots | 5.9 |
| `cog` | Course Over Ground in degrees | 47.5 |
| `vessel_type` | Numeric code for the type of vessel | 70 |
| `vessel_name` | Name of the vessel | OOCL MALAYSIA |
| `length` | Length of the vessel in meters | 366 |
| `width` | Width (beam) of the vessel in meters | 51 |
| `draft` | Draft depth in meters | 14.5 |

Ships typically transmit their position every 1-3 minutes, so a single day of US-wide data contains *millions* of rows. This is exactly the kind of dataset where Polars shines.

## Loading the Data

We have daily AIS data files stored as parquet files. Each file contains all transmissions received on a single day.

Let's start by loading one day of data to understand its structure. We will use `pl.scan_parquet` (lazy mode) because the files are large.

In [ ]:
# Lazy scan -- does not actually read the file yet
lf = pl.scan_parquet("data/2024/ais-2024-01-01.parquet")

lf

Notice that Polars shows us the **schema** (column names and types) without reading the data. This is one of the benefits of lazy evaluation -- we can inspect and build queries before doing any heavy computation.

Let's collect the first few rows to see actual data.

In [ ]:
lf.head(5).collect()

### How big is this data?

Let's count the total number of transmissions and unique ships in this single day.

In [ ]:
summary = (
    lf
    .select(
        pl.col("mmsi").count().alias("total_transmissions"),
        pl.col("mmsi").n_unique().alias("unique_ships"),
    )
    .collect()
)

summary

Over 7 million transmissions from nearly 15,000 ships -- and this is just one day!

This is why we use Polars and parquet files. Loading and processing 7 million rows of a CSV with pandas would be painfully slow.

## Parsing Timestamps

Notice that `base_date_time` is stored as a string (e.g., `"2024-01-01T00:00:03"`). Before we can do any time-based analysis, we need to convert this into a proper datetime type.

In Polars, we use the `.str.to_datetime` method with a format string.

In [ ]:
lf_parsed = lf.with_columns(
    pl.col("base_date_time")
    .str.to_datetime("%Y-%m-%dT%H:%M:%S")
    .alias("timestamp")
)

lf_parsed.head(3).collect()

We now have a `timestamp` column with the proper `Datetime` type. This will allow us to do things like extract hours, compute durations, and filter by time ranges.

## Understanding Vessel Types

The `vessel_type` column uses numeric codes from the [AIS specification](https://coast.noaa.gov/data/marinecadastre/ais/data-dictionary.pdf). The codes we care about most are:

| Code Range | Vessel Type |
|------------|-------------|
| 30-39 | Fishing, Towing, Dredging, Military, Sailing |
| 40-49 | High-speed craft |
| 50-59 | Pilot, Search & Rescue, Tugs, Port Tenders |
| 60-69 | Passenger ships |
| 70-79 | **Cargo ships** |
| 80-89 | Tankers |
| 90-99 | Other (incl. law enforcement) |

For our analysis, we will focus on **cargo ships** (codes 70-79) because these are the vessels that carry goods between ports. When a cargo ship stops somewhere for an extended period, it is very likely at a port loading or unloading goods.

Let's see how many transmissions come from each vessel type category.

In [ ]:
vessel_type_labels = {
    30: "Fishing/Towing/Military",
    40: "High-speed craft",
    50: "Pilot/SAR/Tugs",
    60: "Passenger",
    70: "Cargo",
    80: "Tanker",
    90: "Other",
}

type_counts = (
    lf
    .with_columns(
        # Floor to nearest 10 to group by category
        (pl.col("vessel_type") / 10).floor().cast(pl.Int32).alias("type_category")
    )
    .group_by("type_category")
    .agg(
        pl.col("mmsi").count().alias("transmissions"),
        pl.col("mmsi").n_unique().alias("unique_ships"),
    )
    .sort("type_category")
    .collect()
)

type_counts

### Filtering to cargo ships

From here on, we will work exclusively with cargo ships. Let's create a filtered lazy frame.

In [ ]:
cargo = (
    lf_parsed
    .filter(
        pl.col("vessel_type").is_between(70, 79)
    )
)

cargo_summary = cargo.select(
    pl.col("mmsi").count().alias("transmissions"),
    pl.col("mmsi").n_unique().alias("unique_ships"),
).collect()

cargo_summary

## What Does a Single Ship's Day Look Like?

Before we start aggregating, it is helpful to look at the raw data for one ship. This will help us build intuition about the data's structure.

Let's pick a ship and plot its track for the day.

In [ ]:
cargo.select("mmsi").unique().collect()

In [ ]:
# Pick the first cargo ship in the dataset
first_mmsi = cargo.select(pl.col("mmsi").first()).collect().item()

one_ship = (
    cargo
    .filter(pl.col("mmsi") == first_mmsi)
    .sort("timestamp")
    .collect()
)

print(f"Ship: {one_ship['vessel_name'][0]} (MMSI: {first_mmsi})")
print(f"Transmissions: {one_ship.shape[0]}")
print(f"Time range: {one_ship['timestamp'][0]} to {one_ship['timestamp'][-1]}")
one_ship.head(5)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: plot the ship's track on a map
world.plot(ax=axes[0], color="whitesmoke", edgecolor="black", linewidth=0.5)

track = LineString(
    zip(one_ship["longitude"].to_list(), one_ship["latitude"].to_list())
)
track_gdf = gpd.GeoDataFrame(geometry=[track], crs="EPSG:4326")
track_gdf.plot(ax=axes[0], color="C0", linewidth=1.5)

# Add start and end markers
axes[0].plot(
    one_ship["longitude"][0], one_ship["latitude"][0],
    "o", color="green", markersize=8, label="Start"
)
axes[0].plot(
    one_ship["longitude"][-1], one_ship["latitude"][-1],
    "s", color="red", markersize=8, label="End"
)

# Zoom to the ship's track with some padding
pad = 10.0
axes[0].set_xlim(
    one_ship["longitude"].min() - pad,
    one_ship["longitude"].max() + pad,
)
axes[0].set_ylim(
    one_ship["latitude"].min() - pad,
    one_ship["latitude"].max() + pad,
)
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title(f"Track of {one_ship['vessel_name'][0]}")
axes[0].legend(fontsize=8)

# Right: plot speed over ground through the day
axes[1].plot(
    one_ship["timestamp"].to_numpy(),
    one_ship["sog"].to_numpy(),
    linewidth=0.5
)
axes[1].set_xlabel("Time (UTC)")
axes[1].set_ylabel("Speed (knots)")
axes[1].set_title("Speed Over Ground")

fig.tight_layout()

Notice two things:

1. The track plot shows where the ship traveled throughout the day
2. The speed plot shows periods of movement and rest

When the speed over ground (`sog`) drops near zero, the ship is effectively stationary -- it is either at port, anchored, or drifting.

## Identifying Stopped Ships

We define a ship as **stopped** when its speed over ground is less than 0.5 knots. This is a standard threshold in the maritime literature -- at this speed, a vessel is essentially stationary (even small waves and currents can cause readings of 0.1-0.3 knots on a parked ship).

Our goal: for each cargo ship on this day, count how many transmissions had `sog < 0.5`.

Since ships transmit roughly once per minute, the number of "stopped" transmissions is approximately equal to the number of **minutes** the ship was stopped.

In [ ]:
stopped_minutes = (
    cargo
    .filter(
        pl.col("sog") < 0.5
    )
    .group_by("mmsi")
    .agg(
        pl.col("timestamp").count().alias("n_stopped_obs"),
        pl.col("vessel_name").first(),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .collect()
)

print(f"Total cargo ships: {cargo.select(pl.col('mmsi').n_unique()).collect().item()}")
print(f"Cargo ships with at least 1 stopped observation: {stopped_minutes.shape[0]}")

stopped_minutes.sort("n_stopped_obs", descending=True).head(50)

### Distribution of stopped time

Let's visualize how long ships tend to stop. Since each observation is roughly one minute, we can convert to hours by dividing by 60.

In [ ]:
stopped_hours = stopped_minutes["n_stopped_obs"].to_numpy() / 60

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(stopped_hours, bins=50, edgecolor="white")
ax.axvline(x=6, color="red", linestyle="--", linewidth=2, label="6-hour threshold")
ax.set_xlabel("Hours stopped")
ax.set_ylabel("Number of ships")
ax.set_title("Distribution of Stopped Time for Cargo Ships (Jan 1, 2024)")
ax.legend()

fig.tight_layout()

The histogram shows a clear bimodal pattern: many ships have only a few stopped observations (they were moving most of the day), while a large group was stopped for most of the day (18+ hours). The ships in this second group are almost certainly sitting at port.

This motivates the threshold we will use in our analysis.

## The 6-Hour Rule

We will define a ship as **stopped at port** on a given day if it had a speed below 0.5 knots for **more than 6 hours** (360 observations).

Why 6 hours?

- Loading/unloading cargo takes many hours
- This threshold filters out ships that merely slowed down temporarily (e.g., in a congested channel or waiting briefly at anchor)
- It is conservative enough to give us high confidence that the ship was truly stationary

In [ ]:
stopped_at_port = stopped_minutes.filter(
    pl.col("n_stopped_obs") >= 360  # 360 minutes = 6 hours
)

print(f"Cargo ships stopped for 6+ hours: {stopped_at_port.shape[0]}")
stopped_at_port.sort("n_stopped_obs", descending=True).head(10)

### Plotting the stop locations

Each of these ships was stationary for most of the day. The mean latitude and longitude of their stopped transmissions gives us an approximate location.

Let's plot them on a map.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

ax.scatter(
    stopped_at_port["mean_lon"].to_numpy(),
    stopped_at_port["mean_lat"].to_numpy(),
    s=15, alpha=0.6
)

ax.set_xlim(-140, -50)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Locations of Cargo Ships Stopped 6+ Hours (Jan 1, 2024)")

fig.tight_layout()

Even with just one day, you can already start to see clustering along the US coastlines. The Gulf Coast, East Coast, and West Coast all have visible concentrations of stopped ships. Some ships show up in unexpected places (Hawaii, the Pacific) -- these might be anchored at sea or at smaller facilities.

But how do we systematically identify these clusters? This is where DBSCAN comes in.

## Scaling Up: Multiple Days

One day gives us a good starting point, but ports are easier to identify when we use more data. Let's load the first week of January 2024 and repeat our analysis.

This is where Polars lazy evaluation and `scan_parquet` really shine -- we can scan multiple files at once using a glob pattern.

In [ ]:
import glob

# Get the first 7 days of January
jan_files = sorted(
    glob.glob("data/2024/ais-2024-01-0[1-7].parquet")
)

print(f"Loading {len(jan_files)} files:")
for f in jan_files:
    print(f"  {f}")

In [ ]:
%%time

lf_week = pl.scan_parquet(jan_files)

stopped_week = (
    lf_week
    .with_columns(
        pl.col("base_date_time")
        .str.to_datetime("%Y-%m-%dT%H:%M:%S")
        .alias("timestamp")
    )
    .filter(
        pl.col("vessel_type").is_between(70, 79),
        pl.col("sog") < 0.5,
    )
    .with_columns(
        pl.col("timestamp").dt.date().alias("date")
    )
    .group_by("mmsi", "date")
    .agg(
        pl.col("timestamp").count().alias("n_stopped_obs"),
        pl.col("vessel_name").first(),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .filter(
        pl.col("n_stopped_obs") >= 360
    )
    .collect()
)

print(f"Ship-day combinations stopped 6+ hours: {stopped_week.shape[0]}")
print(f"Unique ships: {stopped_week['mmsi'].n_unique()}")
print(f"Days covered: {stopped_week['date'].n_unique()}")

Notice how fast that was -- Polars processed tens of millions of rows across 7 files in just a few seconds.

A ship that stays at port for multiple days will appear as multiple ship-day entries. For our clustering, we want one location per ship-day (since a ship could move between ports during the week).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

ax.scatter(
    stopped_week["mean_lon"].to_numpy(),
    stopped_week["mean_lat"].to_numpy(),
    s=8, alpha=0.3
)

ax.set_xlim(-140, -50)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Stopped Cargo Ships -- First Week of January 2024")

fig.tight_layout()

With a week of data, the port locations are much clearer. You should see tight clusters along the Gulf Coast (Houston, New Orleans), the East Coast (Norfolk, New York, Savannah), and the West Coast (LA/Long Beach, San Francisco, Seattle).

Now let's formalize this observation using a clustering algorithm.

## Introduction to DBSCAN

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) is a clustering algorithm that is particularly well-suited for our problem.

Unlike K-Means (which requires you to specify the number of clusters in advance), DBSCAN discovers clusters automatically based on the **density** of points.

### How DBSCAN works

DBSCAN has two parameters:

- **`eps`** (epsilon): The maximum distance between two points for them to be considered neighbors
- **`min_samples`**: The minimum number of points needed to form a dense region (a cluster)

The algorithm works as follows:

1. Pick a point that hasn't been visited yet
2. Find all points within `eps` distance of it
3. If there are at least `min_samples` neighbors, start a new cluster
4. Expand the cluster by repeating steps 2-3 for each new neighbor
5. Points that aren't close enough to any cluster are labeled as **noise**

This is ideal for port detection because:

- We don't know how many ports there are in advance
- Ports are dense clusters of stopped ships
- Ships stopped in the open ocean (noise) should be ignored
- Port clusters can be different sizes and shapes

## Projecting Coordinates

Before we can run DBSCAN, we need to deal with a subtlety: our ship locations are in **latitude and longitude** (degrees), but DBSCAN measures distances in whatever units the data is in.

The problem is that one degree of longitude does not equal one degree of latitude in terms of actual distance on the Earth's surface. Near the equator, one degree of longitude is about 111 km, but near the poles it shrinks toward zero.

To fix this, we will **project** our latitude/longitude coordinates into a planar coordinate system where distances are measured in meters. We use the `pyproj` library with the `ESRI:102005` projection (an equidistant conic projection suitable for North America).

In [ ]:
from pyproj import Proj

proj = Proj("ESRI:102005")

A projection converts `(longitude, latitude)` pairs into `(x, y)` coordinates in meters. Let's see an example.

In [ ]:
# Houston is at approximately (29.76, -95.37)
houston_x, houston_y = proj(-95.37, 29.76)
print(f"Houston projected: x={houston_x:,.0f} m, y={houston_y:,.0f} m")

# New Orleans is at approximately (29.95, -90.07)
nola_x, nola_y = proj(-90.07, 29.95)
print(f"New Orleans projected: x={nola_x:,.0f} m, y={nola_y:,.0f} m")

# Distance between them
dist_km = np.sqrt((houston_x - nola_x)**2 + (houston_y - nola_y)**2) / 1000
print(f"\nDistance Houston to New Orleans: {dist_km:.0f} km")

Now let's project all of our stopped ship locations.

In [ ]:
lons = stopped_week["mean_lon"].to_numpy()
lats = stopped_week["mean_lat"].to_numpy()

xs, ys = proj(lons, lats)

# Stack into an (N, 2) array for sklearn
coords = np.column_stack([xs, ys])

print(f"Projected {coords.shape[0]} points")
print(f"X range: {coords[:, 0].min()/1000:.0f} km to {coords[:, 0].max()/1000:.0f} km")
print(f"Y range: {coords[:, 1].min()/1000:.0f} km to {coords[:, 1].max()/1000:.0f} km")

## Running DBSCAN

Now we can run DBSCAN on our projected coordinates. We need to choose values for `eps` and `min_samples`:

- **`eps = 25_000`** (25 km): Ships within 25 km of each other are considered neighbors. This is large enough to capture ships spread across a port area (which can span several kilometers of coastline) but small enough to separate distinct ports.
- **`min_samples = 3`**: A cluster needs at least 3 stopped ship-days. This filters out isolated anchorages while keeping smaller ports.

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=25_000, min_samples=3)
db.fit(coords)

labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()

print(f"Clusters found: {n_clusters}")
print(f"Noise points (not in any cluster): {n_noise}")
print(f"Clustered points: {(labels != -1).sum()}")

### Examining the clusters

Let's add the cluster labels back to our DataFrame and examine the largest clusters.

In [ ]:
clustered = stopped_week.with_columns(
    pl.Series("cluster", labels)
)

cluster_summary = (
    clustered
    .filter(pl.col("cluster") >= 0)  # exclude noise
    .group_by("cluster")
    .agg(
        pl.col("mmsi").count().alias("ship_days"),
        pl.col("mmsi").n_unique().alias("unique_ships"),
        pl.col("mean_lat").mean().alias("center_lat"),
        pl.col("mean_lon").mean().alias("center_lon"),
    )
    .sort("ship_days", descending=True)
)

cluster_summary.head(15)

Look at the center coordinates of the largest clusters and compare them to known port locations:

| Approximate Location | Major Port |
|---------------------|------------|
| (29.9, -90.2) | New Orleans / Mississippi River |
| (37.1, -76.2) | Hampton Roads / Norfolk, VA |
| (49.3, -123.1) | Vancouver, BC |
| (29.7, -95.1) | Houston Ship Channel |
| (29.2, -94.7) | Texas City / Galveston |
| (33.7, -118.2) | Los Angeles / Long Beach |
| (37.8, -122.3) | San Francisco / Oakland |
| (40.6, -74.1) | New York / Newark |
| (47.6, -122.4) | Seattle / Tacoma |

The algorithm has discovered major US ports purely from the patterns in ship movement data!

## Visualizing the Clusters

Let's create a color-coded scatter plot to see the clusters on a map.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

# Plot noise points in gray
noise_mask = labels == -1
ax.scatter(
    lons[noise_mask], lats[noise_mask],
    s=10, c="lightgray", alpha=0.4, label="Noise"
)

# Plot clustered points with colors
cluster_mask = labels >= 0
scatter = ax.scatter(
    lons[cluster_mask], lats[cluster_mask],
    s=15, c=labels[cluster_mask], cmap="tab20", alpha=0.6
)

# Label the top clusters
top_clusters = cluster_summary.head(10)
for row in top_clusters.iter_rows(named=True):
    ax.annotate(
        f"Cluster {row['cluster']}\n({row['unique_ships']} ships)",
        xy=(row["center_lon"], row["center_lat"]),
        fontsize=7,
        fontweight="bold",
        ha="center",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8),
    )

ax.set_xlim(-140, -50)
# ax.set_ylim(25, 32.5)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("DBSCAN Clusters of Stopped Cargo Ships (Jan 1-7, 2024)")

fig.tight_layout()

## Putting Clusters on a Real Map

A scatter plot of longitude vs latitude works, but it would be much nicer to see our clusters overlaid on an actual map. We can use `geopandas` and the Natural Earth coastline data to add geographic context.

In [ ]:
# Create a GeoDataFrame of cluster centroids
centroids = cluster_summary.to_pandas()
centroids["geometry"] = [
    Point(lon, lat)
    for lon, lat in zip(centroids["center_lon"], centroids["center_lat"])
]
centroids_gdf = gpd.GeoDataFrame(centroids, geometry="geometry", crs="EPSG:4326")

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

# Plot the country borders
world.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=0.5)

# Plot all stopped ships colored by cluster
ax.scatter(
    lons[noise_mask], lats[noise_mask],
    s=8, c="lightgray", alpha=0.3, zorder=2
)
ax.scatter(
    lons[cluster_mask], lats[cluster_mask],
    s=12, c=labels[cluster_mask], cmap="tab20", alpha=0.6, zorder=3
)

# Plot centroids as large stars
centroids_gdf.head(15).plot(
    ax=ax, color="red", markersize=100, marker="*",
    edgecolor="black", linewidth=0.5, zorder=4
)

# Focus on the continental US
ax.set_xlim(-130, -65)
ax.set_ylim(24, 52)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Discovered Port Locations (red stars = cluster centers)")

fig.tight_layout()

## Sensitivity Analysis: Choosing `eps`

Our choice of `eps = 25_000` (25 km) was reasonable, but how sensitive are our results to this choice?

Let's try a few different values and see how the number of clusters changes.

In [ ]:
eps_values = [10_000, 15_000, 20_000, 25_000, 30_000, 40_000, 50_000]
results = []

for eps in eps_values:
    db_test = DBSCAN(eps=eps, min_samples=3).fit(coords)
    n_clust = len(set(db_test.labels_)) - (1 if -1 in db_test.labels_ else 0)
    n_noise = (db_test.labels_ == -1).sum()
    results.append({
        "eps_km": eps / 1000,
        "n_clusters": n_clust,
        "n_noise": n_noise,
        "pct_clustered": 100 * (1 - n_noise / len(db_test.labels_))
    })

pl.DataFrame(results)

As `eps` increases:
- Fewer clusters are found (nearby ports get merged)
- More points get assigned to clusters (fewer noise points)

There is no single "correct" value -- the right choice depends on what question you are trying to answer. If you need to distinguish between nearby ports (e.g., Houston vs Galveston), use a smaller `eps`. If you want broad port regions, use a larger one.

## Scaling to a Full Month

Let's see what happens when we use the entire month of January 2024. With more data, the port clusters should become sharper and we may discover smaller ports that only appear with more observations.

In [ ]:
%%time

jan_all_files = sorted(
    glob.glob("data/2024/ais-2024-01-*.parquet")
)
print(f"Loading {len(jan_all_files)} daily files...")

lf_month = pl.scan_parquet(jan_all_files)

stopped_month = (
    lf_month
    .with_columns(
        pl.col("base_date_time")
        .str.to_datetime("%Y-%m-%dT%H:%M:%S")
        .alias("timestamp")
    )
    .filter(
        pl.col("vessel_type").is_between(70, 79),
        pl.col("sog") < 0.5,
    )
    .with_columns(
        pl.col("timestamp").dt.date().alias("date")
    )
    .group_by("mmsi", "date")
    .agg(
        pl.col("timestamp").count().alias("n_stopped_obs"),
        pl.col("vessel_name").first(),
        pl.col("latitude").mean().alias("mean_lat"),
        pl.col("longitude").mean().alias("mean_lon"),
    )
    .filter(
        pl.col("n_stopped_obs") >= 360
    )
    .collect()
)

print(f"Ship-day combinations: {stopped_month.shape[0]}")
print(f"Unique ships: {stopped_month['mmsi'].n_unique()}")

In [ ]:
# Project and cluster
lons_month = stopped_month["mean_lon"].to_numpy()
lats_month = stopped_month["mean_lat"].to_numpy()
xs_month, ys_month = proj(lons_month, lats_month)
coords_month = np.column_stack([xs_month, ys_month])

db_month = DBSCAN(eps=25_000, min_samples=5).fit(coords_month)
labels_month = db_month.labels_

n_clusters_month = len(set(labels_month)) - (1 if -1 in labels_month else 0)
print(f"Clusters: {n_clusters_month}")
print(f"Noise: {(labels_month == -1).sum()}")
print(f"Clustered: {(labels_month >= 0).sum()}")

In [ ]:
# Build summary for the month
clustered_month = stopped_month.with_columns(
    pl.Series("cluster", labels_month)
)

month_summary = (
    clustered_month
    .filter(pl.col("cluster") >= 0)
    .group_by("cluster")
    .agg(
        pl.col("mmsi").count().alias("ship_days"),
        pl.col("mmsi").n_unique().alias("unique_ships"),
        pl.col("mean_lat").mean().alias("center_lat"),
        pl.col("mean_lon").mean().alias("center_lon"),
    )
    .sort("ship_days", descending=True)
)

month_summary.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))

world.plot(ax=ax, color="whitesmoke", edgecolor="black", linewidth=0.5)

noise_month = labels_month == -1
cluster_month = labels_month >= 0

ax.scatter(
    lons_month[noise_month], lats_month[noise_month],
    s=4, c="lightgray", alpha=0.15, zorder=2
)
ax.scatter(
    lons_month[cluster_month], lats_month[cluster_month],
    s=8, c=labels_month[cluster_month], cmap="tab20", alpha=0.4, zorder=3
)

# Star markers for top 20 cluster centers
top20 = month_summary.head(20)
ax.scatter(
    top20["center_lon"].to_numpy(),
    top20["center_lat"].to_numpy(),
    s=150, c="red", marker="*", edgecolor="black",
    linewidth=0.5, zorder=5
)

ax.set_xlim(-130, -65)
ax.set_ylim(24, 52)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Discovered Port Locations -- January 2024 (full month)")

fig.tight_layout()

With a full month of data, the picture is much richer. Smaller ports that were invisible with just one day or one week of data now emerge as distinct clusters.

## Tracking Port Activity Over Time

Now that we can identify ports, let's ask an economic question: **how does the number of ships at each port change over time?**

We can count the number of unique ships stopped at each cluster on each day to create a time series of port activity.

In [ ]:
port_activity = (
    clustered_month
    .filter(pl.col("cluster") >= 0)
    .group_by("cluster", "date")
    .agg(
        pl.col("mmsi").n_unique().alias("ships_at_port")
    )
    .sort("cluster", "date")
)

port_activity.head(20).tail(10)

In [ ]:
# Plot the top 4 ports' activity over January
top4_clusters = month_summary.head(4)["cluster"].to_list()

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

for ax, cluster_id in zip(axes.flatten(), top4_clusters):
    port_data = port_activity.filter(pl.col("cluster") == cluster_id).sort("date")

    row = month_summary.filter(pl.col("cluster") == cluster_id).row(0, named=True)

    ax.bar(
        port_data["date"].to_numpy(),
        port_data["ships_at_port"].to_numpy(),
        width=0.8,
        alpha=0.7
    )

    ax.set_title(
        f"Cluster {cluster_id} ({row['center_lat']:.1f}, {row['center_lon']:.1f})"
    )
    ax.set_ylabel("Ships at port")

axes[1, 0].set_xlabel("Date")
axes[1, 1].set_xlabel("Date")

fig.suptitle("Daily Ships at Port -- January 2024", fontsize=14)
fig.tight_layout()

Notice any weekly patterns? Many ports show dips on weekends when fewer ships are processed. This kind of high-frequency economic data would be impossible to observe with traditional monthly trade statistics.

## Exercises

**Exercise 1**

Repeat the analysis above but for **tanker ships** (vessel types 80-89) instead of cargo ships. Do tankers cluster at the same ports as cargo ships, or do you see different patterns?

*Hint*: You only need to change the `vessel_type` filter. Think about why tanker ports might differ from cargo ports.

In [ ]:
# Your code here

**Exercise 2**

Using the full-month clustered data, find the ship that visited the **most different ports** (clusters) during January 2024. What was its name? How many distinct ports did it visit?

*Hint*: Group by `mmsi`, then count the number of unique cluster values.

In [ ]:
# Your code here

**Exercise 3**

Try using `min_samples=10` and `eps=15_000` (15 km) on the full January data. How does the number of clusters change? Do any of the top ports disappear? Why might a tighter set of parameters be useful?

In [ ]:
# Your code here

**Exercise 4**

Load the first week of **July** 2024 and repeat the full pipeline (filter cargo ships, find 6+ hour stops, project, cluster). Compare the ports and activity levels to what we found in January.

Are there seasonal differences? Are some ports busier in summer vs winter?

In [ ]:
# Your code here

## Summary

In this notebook we:

1. **Loaded and explored** large-scale AIS ship tracking data using Polars
2. **Parsed timestamps**, filtered to cargo ships, and identified vessels that were stopped for more than 6 hours
3. **Projected** latitude/longitude coordinates into a planar system suitable for distance calculations
4. **Applied DBSCAN** clustering to automatically discover port locations from the raw data
5. **Visualized** the results on maps and tracked port activity over time

The key takeaway: with the right tools (Polars for processing, scikit-learn for clustering), we can extract meaningful economic signals from large alternative datasets. The ports we discovered are not hard-coded -- they emerge naturally from the patterns in ship behavior.